# Electricity System Adequacy under the CLEVER Sufficiency Scenario — Horizon 2050

**Objective.**  Assess whether a 100 % renewable + sufficiency electricity system
(as defined by the CLEVER European scenario) can reliably meet demand across
nine interconnected European countries in 2050.

**Pipeline overview:**

1. **Fetch** the CLEVER spreadsheet and EOLES cost assumptions.
2. **Process** the spreadsheet into structured CSV tables.
3. **Build hourly demand** via DemandForge (thermosensitive + EV shaping).
4. **Construct the POMMES model** calibrated on CLEVER capacities, VRE profiles
   from SupplyForge, supplyforge hydro data, and EOLES cost parameters.
5. **Solve** the capacity-expansion + dispatch problem (Gurobi / HiGHS).
6. **Analyse** adequacy (LOLE, ENS), prices, and the optimal capacity mix.

**Requirements:** `clever`, `supplyforge`, `demandforge`, `pommes_craft`, `pommes`
(plus a solver: Gurobi recommended, HiGHS as fallback).


## 1. Configuration


In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import polars as pl

# ── Suppress noisy warnings from POMMES / linopy / solver ──────────
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*shadow price.*")
warnings.filterwarnings("ignore", message=".*Overwriting.*")

# Only show WARNING+ from noisy libraries; keep INFO for clever.*
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
for _lib in ("linopy", "gurobipy", "pommes", "pommes_craft", "xarray", "urllib3"):
    logging.getLogger(_lib).setLevel(logging.WARNING)

# ── Scenario parameters ───────────────────────────────────────────
MODEL_YEAR       = 2050
WEATHER_REF_YEAR = 2024       # SupplyForge / DemandForge reference year (2021 = year with data on GCS)
REF_EV_YEAR      = 2024       # EV profile reference year
SOLVER           = "gurobi"   # "gurobi" or "highs"
ADD_HYDRO        = True       # include hydro (RoR, reservoir, PHS via SupplyForge + PEMMDB)
ADD_INTERCO      = True       # include NTC interconnections

COUNTRIES = ["FR", "DE", "ES", "IT", "GB", "BE", "CH", "AT", "NL"]

print(f"Scenario: CLEVER {MODEL_YEAR} | {len(COUNTRIES)} countries | "
      f"weather ref {WEATHER_REF_YEAR} | solver {SOLVER}")

## 2. Fetch CLEVER data and load EOLES cost assumptions

The CLEVER spreadsheet is downloaded from the public repository.
EOLES techno-economic parameters (CAPEX, fOM, vOM, discount rates,
storage CAPEX) are hardcoded below from the 2026 EOLES CSV files,
eliminating any dependency on local file paths.

In [ ]:
from clever.fetch import fetch_clever_xlsx
from clever.constants import _EOLES_CAPEX_2026, _EOLES_FOM_2026, _EOLES_DISCOUNT_RATE_UNIFORM, _EOLES_VOM_2026, _EOLES_STORAGE_CAPEX_2026, AREA_MAP
xlsx_path = fetch_clever_xlsx(force=False)
print(f"CLEVER spreadsheet: {xlsx_path}  ({xlsx_path.stat().st_size / 1e6:.1f} MB)")

# ── EOLES techno-economic inputs (hardcoded from 2026 EOLES CSVs) ──────
# These dicts replicate exactly what load_eoles_cost_assumptions() would
# return from the 7 CSV files shipped with the EOLES model.  Hardcoding
# them removes any dependency on local file paths.
#
# Structure:  eoles_costs = {
#   "capex":         {tech: €/kW},
#   "fom":           {tech: €/kW/yr},
#   "vom":           {tech: €/MWh},
#   "discount":      {tech: rate (0–1)},
#   "storage_capex": {tech: €/kWh},
# }
eoles_costs = {
    "capex":         _EOLES_CAPEX_2026,
    "fom":           _EOLES_FOM_2026,
    "vom":           _EOLES_VOM_2026,
    "discount":      _EOLES_DISCOUNT_RATE_UNIFORM,
    "storage_capex": _EOLES_STORAGE_CAPEX_2026,
}

print(f"EOLES cost categories: {list(eoles_costs.keys())}")
print(f"  capex:         {len(eoles_costs['capex'])} technologies")
print(f"  fom:           {len(eoles_costs['fom'])} technologies")
print(f"  vom:           {len(eoles_costs['vom'])} technologies")
print(f"  discount:      {len(eoles_costs['discount'])} technologies")
print(f"  storage_capex: {len(eoles_costs['storage_capex'])} technologies")

## 3. Process CLEVER spreadsheet into CSV tables


In [ ]:
from clever.process import process_clever_xlsx

tables = process_clever_xlsx(xlsx_path)

for name, df in tables.items():
    print(f"  {name:15s}  {len(df):>6d} rows  |  columns: {list(df.columns)}")


### 3.1 Inspect CLEVER VRE capacities for modelled countries


In [ ]:
cap = tables["capacity"].copy()
cap = cap[cap["area"].isin(COUNTRIES)]
cap["year_op"] = cap["year_op"].astype(int)

cap_2050 = cap[cap["year_op"] == MODEL_YEAR].copy()
pivot = cap_2050.pivot_table(index="area", columns="conversion_tech", values="value", aggfunc="sum")
display(pivot.round(1).fillna("-"))


## 4. Build hourly electricity demand via DemandForge

DemandForge decomposes annual CLEVER electricity consumption into four
thermosensitive components (baseload, winter heating, summer cooling, EV)
and shapes each one using historical hourly load patterns.


In [ ]:
from clever.demand import (
    read_total_electricity_twh,
    read_end_use_twh,
    compute_targets,
    build_hourly_profiles,
)
from clever import CLEVER_CSV_DIR, DEMAND_DIR
from pathlib import Path

DEMAND_DIR.mkdir(parents=True, exist_ok=True)

# Read CLEVER annual totals and end-use splits
base_totals = read_total_electricity_twh(CLEVER_CSV_DIR)
end_use_csv = CLEVER_CSV_DIR / "clever_end_use_electricity.csv"
end_use_wide = read_end_use_twh(end_use_csv) if end_use_csv.exists() else None

# Compute MWh targets
targets = compute_targets(base_totals, end_use_wide, allow_fallback=True)
targets = targets[targets["area"].isin(COUNTRIES)].copy()
targets = targets[targets["year_op"] == MODEL_YEAR].copy()

print(f"Demand targets for {MODEL_YEAR}:")
display(targets[["area", "total_elec_twh", "heat_twh", "cool_twh", "ev_twh"]].set_index("area").round(2))


### 4.1 Build hourly profiles


In [ ]:
hourly = build_hourly_profiles(
    targets_df=targets,
    out_dir=DEMAND_DIR,
    per_area_files=False,
    tol_rel=1e-2,
    reference_year=WEATHER_REF_YEAR,
    reference_ev_year=REF_EV_YEAR,
)

print(f"Hourly demand: {len(hourly)} rows")
print(f"Countries covered: {sorted(hourly['area'].unique())}")


## 5. Load demand and model inputs


In [ ]:
from clever.runner import load_hourly_total, to_8760, build_demand_dict

demand_wide = load_hourly_total(DEMAND_DIR / "hourly_electricity_demand.csv")
demand_wide = demand_wide[demand_wide["area"].isin(COUNTRIES)].copy()

if MODEL_YEAR is not None:
    demand_wide = demand_wide[demand_wide["year_op"] == MODEL_YEAR].copy()

demand_dict = build_demand_dict(demand_wide)

# Check which countries have demand entries
missing_countries = [c for c in COUNTRIES if c not in demand_dict]
available_countries = [c for c in COUNTRIES if c in demand_dict]

if missing_countries:
    print(f"WARNING: Missing demand entries for {missing_countries}")
    print(f"  -> These countries will be excluded from the model.")
    COUNTRIES_MODELLED = available_countries
else:
    print(f"OK: All {len(COUNTRIES)} countries have demand entries.")
    COUNTRIES_MODELLED = COUNTRIES

print(f"\nDemand dict keys: {sorted(demand_dict.keys())}")
for area, df in sorted(demand_dict.items()):
    peak = float(df["demand"].max())
    energy_twh = float(df["demand"].sum()) / 1e6
    print(f"  {area}: peak={peak:,.0f} MW  |  annual={energy_twh:.1f} TWh")

In [ ]:
from clever.model import (
    read_clever_capacity_csv,
    read_clever_load_factor_csv,
    read_clever_non_enr_csv,
)
from clever import CLEVER_CSV_DIR

capacity_df    = read_clever_capacity_csv(CLEVER_CSV_DIR / "energy_conversion_tech_capacity.csv")
load_factor_df = read_clever_load_factor_csv(CLEVER_CSV_DIR / "energy_conversion_tech_load_factor.csv")
non_enr_df     = read_clever_non_enr_csv(CLEVER_CSV_DIR / "non_enr.csv")

# eoles_costs is already defined in cell 2 (hardcoded EOLES dicts)
assert isinstance(eoles_costs, dict), "eoles_costs not defined — re-run cell 2"
assert set(eoles_costs.keys()) == {"capex", "fom", "vom", "discount", "storage_capex"}, \
    f"Unexpected eoles_costs keys: {set(eoles_costs.keys())}"

print(f"CLEVER inputs loaded:")
print(f"  capacity_df:    {len(capacity_df)} rows")
print(f"  load_factor_df: {len(load_factor_df)} rows")
print(f"  non_enr_df:     {len(non_enr_df)} rows")
print(f"  eoles_costs:    {len(eoles_costs)} cost categories (from cell 2)")

## 6. Ex-ante adequacy diagnostic

Before solving, we check whether the declared capacities plus expansion
headroom can plausibly cover peak demand.  This is a necessary (but not
sufficient) condition for the model to be feasible.


In [ ]:
from clever.adequacy import compute_country_adequacy_metrics

exante_rows = []
for country in COUNTRIES_MODELLED:
    demand_pl = demand_dict.get(country)
    if demand_pl is None:
        continue
    m = compute_country_adequacy_metrics(
        country_code=country,
        model_year=MODEL_YEAR,
        clever_capacity_df=capacity_df,
        clever_non_enr_df=non_enr_df,
        electricity_demand_pl=demand_pl,
    )
    m["country"] = country
    exante_rows.append(m)

exante = pd.DataFrame(exante_rows).set_index("country")
exante = exante[["peak_demand_mw", "dispatchable_existing_mw",
                  "dispatchable_total_max_mw", "vre_credit_mw", "adequacy_margin_mw"]]
display(exante.round(0))

negative = exante[exante["adequacy_margin_mw"] < 0]
if len(negative) > 0:
    print(f"\nWARNING: {len(negative)} countries have negative ex-ante margin.")
    print("The solver will need to invest in Gas / H2 capacity to fill the gap.")
else:
    print("\nAll countries have positive ex-ante margin.")

## 7. Build and solve the POMMES model

The multi-country model includes:
- VRE technologies (Solar, Wind Onshore/Offshore) with SupplyForge hourly profiles
- Dispatchable generation (Gas, Biomass, etc.) from CLEVER TWh data
- Hydro (RoR, reservoir) from SupplyForge chronologies + PEMMDB installed capacities
- Pumped hydro storage (PHS) from PEMMDB
- Endogenous BESS investment (1h and 4h batteries)
- NTC interconnections
- Load shedding at 30,000 EUR/MWh as adequacy backstop

In [ ]:
from clever.model import create_multi_country_model_from_clever
from clever.constants import MANUAL_INTERCONNECTIONS

# Suppress verbose model-building output from POMMES / SupplyForge
for _lib in ("pommes_craft", "supplyforge"):
    logging.getLogger(_lib).setLevel(logging.WARNING)

model = create_multi_country_model_from_clever(
    country_codes=COUNTRIES_MODELLED,
    reference_year_weather=WEATHER_REF_YEAR,
    model_year=MODEL_YEAR,
    clever_capacity_df=capacity_df,
    clever_load_factor_df=load_factor_df,
    clever_non_enr_df=non_enr_df,
    electricity_demand_by_country=demand_dict,
    eoles_costs=eoles_costs,
    interconnections=MANUAL_INTERCONNECTIONS if ADD_INTERCO else None,
    add_interconnections=ADD_INTERCO,
    add_hydro=ADD_HYDRO,
)

# Restore logging
for _lib in ("pommes_craft", "supplyforge"):
    logging.getLogger(_lib).setLevel(logging.INFO)

print(f"Model: {model.name}")
print(f"Hours: {len(model.hours)}  |  Year ops: {model.year_ops}")
print(f"Countries: {COUNTRIES_MODELLED}")

### 7.0.1 Model build verification

Inspect the model object to confirm what technologies and areas were actually
created. This cell is a diagnostic — it should show nonzero VRE, hydro,
dispatchable, and BESS for each country.

In [ ]:
# ── Model build verification ──────────────────────────────────────
print("=" * 80)
print("MODEL BUILD VERIFICATION")
print("=" * 80)

# Inspect areas and their components
try:
    areas = model.areas if hasattr(model, 'areas') else []
    if hasattr(model, '_areas'):
        areas = model._areas
    
    for area_obj in (areas.values() if isinstance(areas, dict) else areas):
        area_name = getattr(area_obj, 'name', str(area_obj))
        components = getattr(area_obj, 'components', getattr(area_obj, '_components', []))
        if isinstance(components, dict):
            components = list(components.values())
        
        conv_techs = []
        stor_techs = []
        for comp in components:
            comp_name = getattr(comp, 'name', str(comp))
            comp_type = type(comp).__name__
            if 'Storage' in comp_type:
                stor_techs.append(comp_name)
            elif 'Conversion' in comp_type:
                cap_max = getattr(comp, 'power_capacity_max', '?')
                conv_techs.append(f"{comp_name} ({cap_max})")
            
        print(f"\n{area_name}:")
        print(f"  Conversion: {', '.join(conv_techs) if conv_techs else 'NONE'}")
        print(f"  Storage:    {', '.join(stor_techs) if stor_techs else 'NONE'}")
        print(f"  Total components: {len(components)}")
except Exception as e:
    print(f"Could not inspect model structure directly: {e}")
    print("This is normal — POMMES model structure may vary.")
    print("Check the INFO-level build logs above for technology additions.")

# Alternative: check parameter tables if available
try:
    if hasattr(model, 'parameter_tables'):
        for key in sorted(model.parameter_tables.keys()):
            df = model.parameter_tables[key]
            print(f"\n  Table: {key} — shape={df.shape}")
except Exception:
    pass

print("\n" + "=" * 80)

### 7.0.2 Parametrise demand flexibility (EV load shifting)

pommes_craft supports demand-side flexibility via the `FlexibleDemand`
component. Each country's Area gets a dedicated `FlexibleDemand` instance
whose hourly profile is derived from the electricity demand.

The cell below adds `FlexibleDemand` components **post-hoc** to the
already-built model. You can change the parameters and re-run this cell +
the solve cell without rebuilding the whole model.

| Parameter | Meaning | Default |
|---|---|---|
| `FLEX_DEMAND_FRACTION` | Share of total hourly demand that is flexible (EV) | 0.08 (8 %) |
| `FLEX_CONSERVATION_HRS` | Hours within which shifted energy must be returned | 6 |
| `FLEX_MAX_MULTIPLIER` | Hourly flex demand can rise to this × nominal flex | 1.15 |
| `FLEX_MIN_MULTIPLIER` | Hourly flex demand can drop to this × nominal flex | 0.85 |
| `FLEX_VARIABLE_COST` | Activation cost of flexibility (EUR/MWh) | 10.0 |

> **Note** — `model.py` now also accepts these kwargs in
> `create_multi_country_model_from_clever()` so you can bake flexibility
> into the model from the start.

In [ ]:
import numpy as np
import polars as pl
from pommes_craft import FlexibleDemand

# ══════════════════════════════════════════════════════════════════════
# FLEXIBILITY PARAMETERS — change these to explore scenarios
# ══════════════════════════════════════════════════════════════════════
FLEX_DEMAND_FRACTION    = 0.08     # fraction of total elec demand that is flexible (EV ~ 8 %)
FLEX_CONSERVATION_HRS   = 6        # shifted energy must be returned within this window (hours)
FLEX_MAX_MULTIPLIER     = 1.15     # max hourly flex demand = 115 % of nominal flex profile
FLEX_MIN_MULTIPLIER     = 0.85     # min hourly flex demand = 85 %  of nominal flex profile
FLEX_RAMP_UP            = np.nan   # MW/h ramp-up limit (NaN = unlimited)
FLEX_RAMP_DOWN          = np.nan   # MW/h ramp-down limit (NaN = unlimited)
FLEX_VARIABLE_COST      = 10.0     # EUR/MWh activation cost of flexibility

# ── Add FlexibleDemand components to each Area ───────────────────────
# Access areas from the model (pommes_craft EnergyModel stores them)
areas = model.areas if hasattr(model, 'areas') else {}
if not areas and hasattr(model, '_areas'):
    areas = model._areas
if isinstance(areas, list):
    areas = {getattr(a, 'name', str(a)): a for a in areas}

added_count = 0
with model.context():
    for country_code, area in areas.items():
        # Get hourly demand profile for this country
        demand_pl = demand_dict.get(country_code)
        if demand_pl is None:
            print(f"  SKIP {country_code}: no demand data in demand_dict")
            continue

        # Extract hourly demand values
        if "demand" in demand_pl.columns:
            demand_vals = demand_pl["demand"].to_list()
        else:
            numeric_cols = [c for c in demand_pl.columns if c not in ("hour", "year_op")]
            demand_vals = demand_pl[numeric_cols[0]].to_list()

        n_hours = len(demand_vals)

        # Build hourly profiles
        flex_demand_profile = [d * FLEX_DEMAND_FRACTION for d in demand_vals]
        flex_max_profile    = [d * FLEX_MAX_MULTIPLIER  for d in flex_demand_profile]
        flex_min_profile    = [d * FLEX_MIN_MULTIPLIER  for d in flex_demand_profile]

        hours    = list(range(n_hours))
        year_ops = [MODEL_YEAR] * n_hours

        flex_demand_df = pl.DataFrame({"hour": hours, "year_op": year_ops, "demand": flex_demand_profile})
        flex_max_df    = pl.DataFrame({"hour": hours, "year_op": year_ops, "max_demand": flex_max_profile})
        flex_min_df    = pl.DataFrame({"hour": hours, "year_op": year_ops, "min_demand": flex_min_profile})

        # Remove any previously added FlexibleDemand (in case cell is re-run)
        components = getattr(area, 'components', getattr(area, '_components', {}))
        if isinstance(components, dict):
            old_flex = [k for k in components if 'flexibility' in k.lower() or 'flex' in k.lower()]
            for k in old_flex:
                del components[k]
        elif isinstance(components, list):
            area._components = [c for c in components
                                if 'flexibility' not in getattr(c, 'name', '').lower()]

        area.add_component(
            FlexibleDemand(
                name=f"ev_flexibility_{country_code}",
                resource="electricity",
                demand=flex_demand_df,
                conservation_hrs=FLEX_CONSERVATION_HRS,
                ramp_up=FLEX_RAMP_UP,
                ramp_down=FLEX_RAMP_DOWN,
                max_demand=flex_max_df,
                min_demand=flex_min_df,
                variable_cost=FLEX_VARIABLE_COST,
            )
        )
        added_count += 1

        total_twh = sum(demand_vals) / 1e6
        flex_twh  = total_twh * FLEX_DEMAND_FRACTION
        print(f"  {country_code}: FlexibleDemand added — {flex_twh:.1f} TWh flexible / {total_twh:.1f} TWh total")

print(f"\nFlexibleDemand activated for {added_count} countries")
print(f"  Fraction: {FLEX_DEMAND_FRACTION:.0%} | Conservation: {FLEX_CONSERVATION_HRS}h")
print(f"  Range: [{FLEX_MIN_MULTIPLIER:.0%}, {FLEX_MAX_MULTIPLIER:.0%}] of flex profile")
print(f"  Variable cost: {FLEX_VARIABLE_COST} EUR/MWh")

### 7.0.3 Review & adjust CCGT H2 capacity expansion bounds

The optimizer can invest in new CCGT H2 (hydrogen-fired gas turbines) up to
a capacity investment cap. In the baseline run the cap was not binding, yet
1.79 TWh of load shedding persists — suggesting temporal rather than pure
capacity bottlenecks.

This cell inspects the H2-related technologies in the model components and
optionally overrides their `investment_max` directly on the
`ConversionTechnology` objects (not on `model.dataset`, which is rebuilt
from scratch by the solve pipeline).

> **Important** — modifying `model.dataset` does NOT propagate to the
> solver. The solve pipeline calls `model.to_pommes_model()` which
> regenerates the input dataset from components. All overrides must go
> through component attributes.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CCGT H2 CAPACITY EXPANSION — review and adjust via components
# ══════════════════════════════════════════════════════════════════════
CCGT_H2_INV_MAX = None  # Set to a value in MW (e.g. 80_000) to override, or None to keep current

# ── Inspect all conversion components across areas ───────────────────
areas = model.areas if hasattr(model, 'areas') else {}
if not areas and hasattr(model, '_areas'):
    areas = model._areas
if isinstance(areas, list):
    areas = {getattr(a, 'name', str(a)): a for a in areas}

print("CCGT H2 / Gas technology inventory")
print("=" * 72)

for country_code, area in sorted(areas.items()):
    components = getattr(area, 'components', getattr(area, '_components', {}))
    if isinstance(components, list):
        components = {getattr(c, 'name', str(c)): c for c in components}

    h2_techs = {k: v for k, v in components.items()
                if any(tag in k.lower() for tag in ("h2", "hydrogen", "ccgt", "gas"))}

    if h2_techs:
        print(f"\n  {country_code}:")
        for tech_name, tech in h2_techs.items():
            inv_max = getattr(tech, 'investment_max', getattr(tech, 'invest_max', 'N/A'))
            cap_max = getattr(tech, 'capacity_max', getattr(tech, 'p_max', 'N/A'))
            print(f"    {tech_name}: inv_max={inv_max}, cap_max={cap_max}")

            # Override if requested
            if CCGT_H2_INV_MAX is not None:
                for attr in ('investment_max', 'invest_max'):
                    if hasattr(tech, attr):
                        old = getattr(tech, attr)
                        setattr(tech, attr, CCGT_H2_INV_MAX)
                        print(f"      -> OVERRIDDEN {attr}: {old} -> {CCGT_H2_INV_MAX} MW")
                        break

if CCGT_H2_INV_MAX is not None:
    print(f"\nCCGT H2 investment cap set to {CCGT_H2_INV_MAX:,.0f} MW across all countries")
else:
    print("\nCCGT_H2_INV_MAX = None -> keeping current bounds (inspect only)")

### 7.1 Solve


In [ ]:
from clever.runner import run_model_without_ramping, build_solver_options
from clever import RESULTS_DIR

solver_options = build_solver_options(solver_name=SOLVER)

# Suppress verbose solver output during solve
logging.getLogger("linopy").setLevel(logging.ERROR)
logging.getLogger("gurobipy").setLevel(logging.ERROR)
logging.getLogger("pommes").setLevel(logging.ERROR)

DIAG_DIR = RESULTS_DIR / "diagnostics"
print(f"Solver: {SOLVER} | Diagnostics: {DIAG_DIR}")

linopy_model = run_model_without_ramping(
    model=model,
    solver_name=SOLVER,
    solver_options=solver_options,
    year_op=MODEL_YEAR,
    write_lp=False,
    diagnostics_dir=DIAG_DIR,
)

# Restore logging after solve
logging.getLogger("linopy").setLevel(logging.WARNING)
logging.getLogger("gurobipy").setLevel(logging.WARNING)
logging.getLogger("pommes").setLevel(logging.WARNING)

print(f"Model solved successfully.")
print(f"Objective value: {linopy_model.objective.value:,.0f} EUR")

## 8. Extract results


In [ ]:
from clever.runner import (
    export_prices,
    export_conversion_capacity,
    export_storage_capacity,
    export_storage_power_capacity,
)

prices_df    = export_prices(model, MODEL_YEAR)
conv_cap_df  = export_conversion_capacity(model, MODEL_YEAR)
stor_cap_df  = export_storage_capacity(model, MODEL_YEAR)
stor_pow_df  = export_storage_power_capacity(model, MODEL_YEAR)

print(f"Prices: {len(prices_df)} rows")
print(f"Conversion capacities: {len(conv_cap_df)} rows")
print(f"Storage energy capacities: {len(stor_cap_df)} rows")
print(f"Storage power capacities: {len(stor_pow_df)} rows")


## 9. Adequacy analysis

Three key indicators:
1. **LOLE** (Loss of Load Expectation): number of hours where load shedding occurs.
2. **ENS** (Energy Not Served): total MWh curtailed by load shedding.
3. **Adequacy margin**: fraction of hours without any load shedding.


In [ ]:
import xarray as xr
from pathlib import Path
from clever.constants import DEFAULT_LOAD_SHEDDING_COST

SHEDDING_THRESHOLD = DEFAULT_LOAD_SHEDDING_COST * 0.9   # 27,000 EUR/MWh
DIAG_DIR = Path("results/diagnostics")

# ── Load solution & dual datasets ─────────────────────────────────
sol_path = DIAG_DIR / f"solution_{MODEL_YEAR}.nc"
dual_path = DIAG_DIR / f"dual_{MODEL_YEAR}.nc"
inp_path = DIAG_DIR / f"input_dataset_{MODEL_YEAR}.nc"

sol_ds  = xr.open_dataset(sol_path)  if sol_path.exists()  else None
dual_ds = xr.open_dataset(dual_path) if dual_path.exists() else None
inp_ds  = xr.open_dataset(inp_path)  if inp_path.exists()  else None

# ── Extract load-shedding from solution.nc ────────────────────────
# Exact variable name: operation_load_shedding_power
# Dims: (area, hour, resource, year_op)
LS_VAR = "operation_load_shedding_power"

adequacy_results = {}
for area in COUNTRIES_MODELLED:
    result = {"area": area}

    # --- ENS from solution.nc (preferred) ---
    if sol_ds is not None and LS_VAR in sol_ds:
        ls_da = sol_ds[LS_VAR]
        try:
            ls_area = ls_da.sel(area=area)
            # Select electricity resource if dimension exists
            if "resource" in ls_area.dims:
                for r in ls_area.coords["resource"].values:
                    if "electr" in str(r).lower():
                        ls_area = ls_area.sel(resource=r)
                        break
            if "year_op" in ls_area.dims:
                ls_area = ls_area.sel(year_op=MODEL_YEAR)
            vals = ls_area.values.flatten()
            ens_mwh = float(vals[vals > 0].sum())
            lole_hrs = int((vals > 0).sum())
            peak_ls = float(vals.max())
            result["ens_gwh"] = ens_mwh / 1000
            result["lole_hours"] = lole_hrs
            result["peak_shedding_mw"] = peak_ls
            result["ens_source"] = "solution.nc"
        except Exception as e:
            result["ens_gwh"] = None
            result["error"] = str(e)
    else:
        result["ens_gwh"] = None
        result["ens_source"] = "not available"

    # --- Adequacy dual from dual.nc ---
    DUAL_VAR = "operation_adequacy_constraint"
    if dual_ds is not None and DUAL_VAR in dual_ds:
        try:
            dual_da = dual_ds[DUAL_VAR]
            dual_area = dual_da.sel(area=area)
            if "resource" in dual_area.dims:
                for r in dual_area.coords["resource"].values:
                    if "electr" in str(r).lower():
                        dual_area = dual_area.sel(resource=r)
                        break
            if "year_op" in dual_area.dims:
                dual_area = dual_area.sel(year_op=MODEL_YEAR)
            dvals = dual_area.values.flatten()
            result["dual_mean"] = float(dvals.mean())
            result["dual_max"]  = float(dvals.max())
            result["dual_gt1k"] = int((dvals > 1000).sum())
            result["dual_at_voll"] = int((dvals >= DEFAULT_LOAD_SHEDDING_COST * 0.99).sum())
        except Exception:
            pass

    # --- Price-based cross-check ---
    area_prices = prices_df[prices_df["area"] == area].copy()
    if not area_prices.empty:
        shedding_hours = area_prices["price"] >= SHEDDING_THRESHOLD
        result["price_lole_hours"] = int(shedding_hours.sum())
        result["max_price"] = float(area_prices["price"].max())
        result["mean_price"] = float(area_prices["price"].mean())

    adequacy_results[area] = result

# ── Display summary ───────────────────────────────────────────────
print("=" * 90)
print("ADEQUACY RESULTS  —  CLEVER Sufficiency Scenario 2050")
print(f"  VoLL = {DEFAULT_LOAD_SHEDDING_COST:,.0f} EUR/MWh")
print("=" * 90)
print(f"{'Country':>8}  {'ENS (GWh)':>10}  {'LOLE (h)':>8}  {'Peak LS':>10}  "
      f"{'Dual max':>10}  {'Dual≥VoLL':>10}  {'Source':>12}")
print("-" * 90)

total_ens = 0
total_lole = 0
for area in COUNTRIES_MODELLED:
    r = adequacy_results[area]
    ens = r.get("ens_gwh")
    lole = r.get("lole_hours", "—")
    peak = r.get("peak_shedding_mw", "—")
    dmax = r.get("dual_max", "—")
    dvoll = r.get("dual_at_voll", "—")
    src = r.get("ens_source", "—")

    ens_str = f"{ens:10.1f}" if ens is not None else "       N/A"
    lole_str = f"{lole:>8}" if isinstance(lole, int) else f"{lole:>8}"
    peak_str = f"{peak:10,.0f}" if isinstance(peak, (int, float)) else f"{peak:>10}"
    dmax_str = f"{dmax:10,.0f}" if isinstance(dmax, (int, float)) else f"{dmax:>10}"
    dvoll_str = f"{dvoll:>10}" if isinstance(dvoll, int) else f"{dvoll:>10}"

    print(f"{area:>8}  {ens_str}  {lole_str}  {peak_str}  {dmax_str}  {dvoll_str}  {src:>12}")

    if ens is not None:
        total_ens += ens
    if isinstance(lole, int):
        total_lole += lole

print("-" * 90)
print(f"{'TOTAL':>8}  {total_ens:10.1f}  {total_lole:>8}")
print()
print("Key: ENS = Energy Not Served | LOLE = Loss of Load Expectation (hours)")
print(f"     Dual≥VoLL = hours where adequacy shadow price hits {DEFAULT_LOAD_SHEDDING_COST:,.0f} EUR/MWh cap")


## 10. Price analysis

Marginal electricity prices from the optimisation reflect the short-run
equilibrium between supply and demand at each hour. In a renewable-dominated
system like CLEVER 2050, prices are near zero during high VRE output and rise
sharply during periods of scarcity (low wind, no sun, storage depleted).

Key benchmarks for sanity-checking results:
- **< 50 EUR/MWh average**: typical for a well-supplied VRE system with adequate flexibility
- **50–150 EUR/MWh**: moderately tight system, possibly insufficient storage or interconnection
- **> 500 EUR/MWh**: structural supply shortage — check for missing technologies or data gaps

In [ ]:
price_stats = (
    prices_df.groupby("area")["value"]
    .agg(["mean", "median", "std", "min", "max"])
    .round(2)
)
price_stats.columns = ["mean_EUR", "median_EUR", "std_EUR", "min_EUR", "max_EUR"]
display(price_stats)

print(f"\nSystem average price: {prices_df['value'].mean():.2f} EUR/MWh")


### 10.1 Price diagnostics — detecting extreme prices

If average prices exceed ~150 EUR/MWh, the model likely faces structural
supply shortage. The cell below flags potential issues by comparing prices
against the load-shedding cost threshold and computing scarcity indicators.

In [ ]:
from clever.constants import DEFAULT_LOAD_SHEDDING_COST

# ── Price diagnostic summary ────────────────────────────────────
print("=" * 72)
print("PRICE DIAGNOSTIC SUMMARY")
print("=" * 72)

for area in sorted(prices_df["area"].unique()):
    ap = prices_df[prices_df["area"] == area]["value"]
    mean_p = ap.mean()
    median_p = ap.median()
    max_p = ap.max()
    hours_scarcity = (ap >= DEFAULT_LOAD_SHEDDING_COST).sum()
    hours_high = (ap >= 500).sum()
    hours_zero = (ap <= 0.01).sum()

    status = "OK"
    if mean_p > 500:
        status = "CRITICAL — extreme scarcity (check VRE profiles / dispatchable capacity)"
    elif mean_p > 150:
        status = "WARNING — elevated prices (possible supply tightness)"
    elif mean_p < 10:
        status = "INFO — very low prices (possible oversupply)"

    print(f"\n{area}:")
    print(f"  Mean: {mean_p:>10.2f} EUR/MWh  |  Median: {median_p:>10.2f} EUR/MWh")
    print(f"  Max:  {max_p:>10.2f} EUR/MWh  |  Scarcity hours (>= {DEFAULT_LOAD_SHEDDING_COST:.0f}): {hours_scarcity}")
    print(f"  Hours > 500 EUR: {hours_high}  |  Hours ~ 0 EUR: {hours_zero}")
    print(f"  Status: {status}")

print("\n" + "=" * 72)

### 10.2 Price distribution by country

Histogram of hourly prices reveals the typical bimodal structure of a
renewable-dominated system: a large cluster near zero (VRE marginal cost)
and a thinner tail at higher values (scarcity / thermal marginal cost).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

areas = sorted(prices_df["area"].unique())
n_areas = len(areas)
cols = min(3, n_areas)
rows = (n_areas + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows), squeeze=False)

for idx, area in enumerate(areas):
    ax = axes[idx // cols][idx % cols]
    ap = prices_df[prices_df["area"] == area]["value"].values

    # Clip for histogram (exclude extreme outliers for readability)
    clip_max = min(float(np.percentile(ap, 99.5)), 1000)
    ap_clipped = ap[ap <= clip_max]

    ax.hist(ap_clipped, bins=80, color="#1976D2", alpha=0.7, edgecolor="white", linewidth=0.3)
    ax.axvline(np.mean(ap), color="red", linestyle="--", linewidth=1.2, label=f"Mean={np.mean(ap):.0f}")
    ax.axvline(np.median(ap), color="orange", linestyle=":", linewidth=1.2, label=f"Median={np.median(ap):.0f}")
    ax.set_title(f"{area} — hourly prices", fontsize=11)
    ax.set_xlabel("EUR/MWh")
    ax.set_ylabel("Hours")
    ax.legend(fontsize=8)

# Hide unused subplots
for idx in range(n_areas, rows * cols):
    axes[idx // cols][idx % cols].set_visible(False)

plt.suptitle("Hourly price distributions by country", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 10.3 Weekly average prices

Weekly averaging smooths out hourly volatility and reveals seasonal
pricing patterns driven by renewable availability and demand seasonality.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

for area in sorted(prices_df["area"].unique()):
    ap = prices_df[prices_df["area"] == area].sort_values("hour")
    weekly = ap.groupby(ap["hour"] // 168)["value"].mean()
    ax.plot(weekly.index, weekly.values, label=area, linewidth=1.2)

ax.set_xlabel("Week of year")
ax.set_ylabel("Average price (EUR/MWh)")
ax.set_title("Weekly average electricity prices by country")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Price correlation matrix ─────────────────────────────────────
price_wide = prices_df.pivot_table(index="hour", columns="area", values="value")
if price_wide.shape[1] > 1:
    corr = price_wide.corr().round(3)
    print("\nPrice correlation matrix (hourly):")
    display(corr)

## 11. Optimal capacity mix


In [ ]:
if not conv_cap_df.empty:
    cap_pivot = conv_cap_df.pivot_table(
        index="area", columns="name", values="value", aggfunc="sum"
    ).fillna(0).round(0)
    print("Conversion technology capacities (MW):")
    display(cap_pivot)

if not stor_pow_df.empty:
    stor_pivot = stor_pow_df.pivot_table(
        index="area", columns="name", values="value", aggfunc="sum"
    ).fillna(0).round(0)
    print("\nStorage power capacities (MW):")
    display(stor_pivot)

if not stor_cap_df.empty:
    stor_e_pivot = stor_cap_df.pivot_table(
        index="area", columns="name", values="value", aggfunc="sum"
    ).fillna(0).round(0)
    print("\nStorage energy capacities (MWh):")
    display(stor_e_pivot)


## 12. VRE Spillage / Curtailment Analysis

**Supervisor question**: *"Spillage pour chaque année / Est-ce que mon curtailment est excessif ?"*

We extract `operation_spillage_power` from `solution.nc` (dims: area, hour, resource, year_op).
A system with zero spillage and high load shedding is capacity-short, not over-supplied.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

SPILL_VAR = "operation_spillage_power"

if sol_ds is not None and SPILL_VAR in sol_ds:
    spill_da = sol_ds[SPILL_VAR]
    print(f"Spillage variable: {SPILL_VAR}")
    print(f"  Dims: {spill_da.dims}, Shape: {spill_da.shape}")

    # Per-country annual spillage
    print("\n── Annual VRE Spillage by Country ──")
    spill_by_country = {}
    for area in COUNTRIES_MODELLED:
        try:
            sp = spill_da.sel(area=area)
            if "resource" in sp.dims:
                for r in sp.coords["resource"].values:
                    if "electr" in str(r).lower():
                        sp = sp.sel(resource=r); break
            if "year_op" in sp.dims:
                sp = sp.sel(year_op=MODEL_YEAR)
            vals = sp.values.flatten()
            total_gwh = vals.sum() / 1000
            peak_mw = vals.max()
            hours_spill = (vals > 0).sum()
            spill_by_country[area] = total_gwh
            print(f"  {area}: {total_gwh:8.1f} GWh  |  {hours_spill:5d} hours  |  peak {peak_mw:,.0f} MW")
        except Exception as e:
            print(f"  {area}: error — {e}")
            spill_by_country[area] = 0

    total_spill = sum(spill_by_country.values())
    print(f"  TOTAL: {total_spill:,.1f} GWh")

    # Compare with total demand
    total_demand_twh = sum(demand_dict[a]["demand"].sum() / 1e6 for a in COUNTRIES_MODELLED if a in demand_dict)
    curtailment_pct = total_spill / (total_demand_twh * 1000) * 100
    print(f"\n  Curtailment ratio: {curtailment_pct:.2f}% of total demand ({total_demand_twh:.1f} TWh)")
    if curtailment_pct < 1:
        print("  → Very low curtailment. System is capacity-short, NOT over-supplied with VRE.")
    elif curtailment_pct < 5:
        print("  → Moderate curtailment. Normal for high-VRE systems.")
    else:
        print("  → High curtailment. Consider additional storage or interconnection capacity.")

    # ── Time-series of hourly spillage (system-wide) ──────────────
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={"height_ratios": [3, 1]})

    # Stacked area by country
    for area in COUNTRIES_MODELLED:
        try:
            sp = spill_da.sel(area=area)
            if "resource" in sp.dims:
                for r in sp.coords["resource"].values:
                    if "electr" in str(r).lower():
                        sp = sp.sel(resource=r); break
            if "year_op" in sp.dims:
                sp = sp.sel(year_op=MODEL_YEAR)
            vals = sp.values.flatten()
            if vals.sum() > 0:
                axes[0].plot(range(len(vals)), vals, label=area, alpha=0.7, linewidth=0.5)
        except:
            pass

    axes[0].set_ylabel("Spillage (MW)")
    axes[0].set_title("Hourly VRE Spillage by Country")
    axes[0].legend(ncol=3, fontsize=8)
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else f"{x:.0f}"))

    # Bar chart of annual spillage
    countries = list(spill_by_country.keys())
    values = [spill_by_country[c] for c in countries]
    colors = plt.cm.Set2(range(len(countries)))
    axes[1].bar(countries, values, color=colors)
    axes[1].set_ylabel("Annual spillage (GWh)")
    axes[1].set_title("Total Annual VRE Spillage by Country")

    plt.tight_layout()
    plt.savefig(DIAG_DIR / "spillage_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print(f"WARNING: {SPILL_VAR} not found in solution.nc")
    if sol_ds is not None:
        spill_candidates = [v for v in sol_ds.data_vars if "spill" in v.lower()]
        print(f"  Candidates: {spill_candidates}")


## 13. LOLE & Load Shedding Detailed Analysis

**Supervisor questions**:
- *"LOLE / Afficher la loss of load → 30k EUR"*
- *"Combien d'heures de loss of load ?"*

We visualise when and where load shedding occurs, and show the shadow price
at the VoLL cap (30,000 EUR/MWh).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

LS_VAR = "operation_load_shedding_power"
DUAL_VAR = "operation_adequacy_constraint"

if sol_ds is not None and LS_VAR in sol_ds:
    fig, axes = plt.subplots(3, 1, figsize=(16, 14), gridspec_kw={"height_ratios": [2, 2, 1]})

    # ── Panel 1: Load-shedding heatmap (hour-of-day vs day-of-year) ──
    # Aggregate across all countries
    ls_da = sol_ds[LS_VAR]
    all_ls = np.zeros(8760)
    for area in COUNTRIES_MODELLED:
        try:
            ls_area = ls_da.sel(area=area)
            if "resource" in ls_area.dims:
                for r in ls_area.coords["resource"].values:
                    if "electr" in str(r).lower():
                        ls_area = ls_area.sel(resource=r); break
            if "year_op" in ls_area.dims:
                ls_area = ls_area.sel(year_op=MODEL_YEAR)
            vals = ls_area.values.flatten()[:8760]
            all_ls[:len(vals)] += vals
        except:
            pass

    # Reshape to (365, 24)
    ls_matrix = all_ls[:8760].reshape(365, 24) if len(all_ls) >= 8760 else all_ls.reshape(-1, 24)
    im = axes[0].imshow(ls_matrix.T, aspect="auto", cmap="Reds", origin="lower",
                         extent=[1, 365, 0, 24])
    axes[0].set_xlabel("Day of year")
    axes[0].set_ylabel("Hour of day")
    axes[0].set_title("System-wide load shedding (MW) — hour × day heatmap")
    plt.colorbar(im, ax=axes[0], label="Load shedding (MW)", shrink=0.8)

    # ── Panel 2: Per-country LOLE timeline ──
    for area in COUNTRIES_MODELLED:
        try:
            ls_area = ls_da.sel(area=area)
            if "resource" in ls_area.dims:
                for r in ls_area.coords["resource"].values:
                    if "electr" in str(r).lower():
                        ls_area = ls_area.sel(resource=r); break
            if "year_op" in ls_area.dims:
                ls_area = ls_area.sel(year_op=MODEL_YEAR)
            vals = ls_area.values.flatten()[:8760]
            if vals.max() > 0:
                axes[1].plot(range(len(vals)), vals, label=area, alpha=0.8, linewidth=0.5)
        except:
            pass

    axes[1].set_ylabel("Load shedding (MW)")
    axes[1].set_xlabel("Hour of year")
    axes[1].set_title("Per-country load shedding timeline")
    axes[1].legend(ncol=3, fontsize=8)

    # ── Panel 3: LOLE hours bar chart ──
    lole_data = {}
    for area in COUNTRIES_MODELLED:
        r = adequacy_results.get(area, {})
        lole_data[area] = r.get("lole_hours", 0)

    countries = list(lole_data.keys())
    hours = [lole_data[c] for c in countries]
    colors = ["#d32f2f" if h > 100 else "#ff9800" if h > 10 else "#4caf50" for h in hours]
    axes[2].bar(countries, hours, color=colors)
    axes[2].set_ylabel("LOLE (hours)")
    axes[2].set_title("Loss of Load Expectation by Country")
    for i, (c, h) in enumerate(zip(countries, hours)):
        if h > 0:
            axes[2].text(i, h + 5, str(h), ha="center", fontsize=9, fontweight="bold")

    # Horizontal line at typical adequacy standard (3h)
    axes[2].axhline(y=3, color="green", linestyle="--", linewidth=1, label="3h standard")
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(DIAG_DIR / "lole_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

# ── Dual price at VoLL ──
if dual_ds is not None and DUAL_VAR in dual_ds:
    print("\n── Adequacy shadow price at VoLL (30,000 EUR/MWh) ──")
    dual_da = dual_ds[DUAL_VAR]
    for area in COUNTRIES_MODELLED:
        try:
            d = dual_da.sel(area=area)
            if "resource" in d.dims:
                for r in d.coords["resource"].values:
                    if "electr" in str(r).lower():
                        d = d.sel(resource=r); break
            if "year_op" in d.dims:
                d = d.sel(year_op=MODEL_YEAR)
            dvals = d.values.flatten()
            at_voll = (dvals >= 30000 * 0.99).sum()
            gt_1k = (dvals > 1000).sum()
            mean_d = dvals.mean()
            print(f"  {area}: {at_voll:4d}h at VoLL cap  |  {gt_1k:4d}h > 1000 EUR  |  mean dual = {mean_d:,.0f} EUR/MWh")
        except Exception as e:
            print(f"  {area}: error — {e}")


## 14. Sector Coupling: Electricity ↔ Hydrogen

**Supervisor question**: *"Couplage"*

Analyse the bidirectional electricity-hydrogen coupling: electrolysis
(electricity → H2) and CCGT H2 (H2 → electricity). Key questions:
- How much H2 is produced / consumed?
- What is the utilisation rate of H2 plants?
- Is H2 storage sufficient to bridge stress periods?

In [ ]:
import matplotlib.pyplot as plt

CONV_VAR = "operation_conversion_power"

if sol_ds is not None and CONV_VAR in sol_ds:
    conv_da = sol_ds[CONV_VAR]
    print(f"Conversion power variable: {CONV_VAR}")
    print(f"  Dims: {conv_da.dims}")

    # Find H2-related technologies
    tech_dim = None
    for dim in conv_da.dims:
        coords = [str(c) for c in conv_da.coords[dim].values]
        h2_techs = [c for c in coords if "h2" in c.lower() or "electrol" in c.lower() or "hydrogen" in c.lower()]
        if h2_techs:
            tech_dim = dim
            print(f"\n  H2 technologies in '{dim}': {h2_techs}")
            break

    if tech_dim:
        all_techs = [str(c) for c in conv_da.coords[tech_dim].values]
        h2_techs = [t for t in all_techs if "h2" in t.lower() or "electrol" in t.lower() or "hydrogen" in t.lower()]

        fig, axes = plt.subplots(len(h2_techs), 1, figsize=(16, 4 * len(h2_techs)), squeeze=False)

        for idx, tech in enumerate(h2_techs):
            ax = axes[idx, 0]
            total_by_country = {}
            for area in COUNTRIES_MODELLED:
                try:
                    p = conv_da.sel({tech_dim: tech, "area": area})
                    if "resource" in p.dims:
                        for r in p.coords["resource"].values:
                            if "electr" in str(r).lower():
                                p = p.sel(resource=r); break
                    if "year_op" in p.dims:
                        p = p.sel(year_op=MODEL_YEAR)
                    vals = p.values.flatten()[:8760]
                    total_twh = abs(vals).sum() / 1e6
                    total_by_country[area] = total_twh
                    if abs(vals).max() > 0:
                        ax.plot(range(len(vals)), vals, label=f"{area} ({total_twh:.1f} TWh)", alpha=0.7, linewidth=0.5)
                except:
                    pass

            ax.set_title(f"{tech} — hourly output by country")
            ax.set_ylabel("Power (MW)")
            ax.set_xlabel("Hour")
            ax.legend(ncol=3, fontsize=7)
            total = sum(total_by_country.values())
            ax.text(0.02, 0.95, f"System total: {total:.1f} TWh", transform=ax.transAxes,
                    fontsize=10, verticalalignment="top", bbox=dict(boxstyle="round", facecolor="wheat"))

        plt.tight_layout()
        plt.savefig(DIAG_DIR / "sector_coupling_h2.png", dpi=150, bbox_inches="tight")
        plt.show()

    # ── H2 storage ──
    STOR_VAR = "operation_storage_level"
    if STOR_VAR in sol_ds:
        stor_da = sol_ds[STOR_VAR]
        print("\n── H2 Storage ──")
        for dim in stor_da.dims:
            coords = [str(c) for c in stor_da.coords[dim].values]
            h2_stor = [c for c in coords if "h2" in c.lower() or "hydrogen" in c.lower()]
            if h2_stor:
                print(f"  H2 storage techs in '{dim}': {h2_stor}")
                for st in h2_stor:
                    for area in COUNTRIES_MODELLED:
                        try:
                            soc = stor_da.sel({dim: st, "area": area})
                            if "year_op" in soc.dims:
                                soc = soc.sel(year_op=MODEL_YEAR)
                            vals = soc.values.flatten()
                            if vals.max() > 0:
                                print(f"    {area}/{st}: max SoC = {vals.max():,.0f} MWh, "
                                      f"min SoC = {vals.min():,.0f} MWh, "
                                      f"mean = {vals.mean():,.0f} MWh")
                        except:
                            pass
else:
    print(f"WARNING: {CONV_VAR} not found in solution.nc")


## 15. Demand Flexibility Results (EV Load Shifting)

**Supervisor questions**:
- *"Demande flexible des véhicules électriques"*
- *"Flexibilité de la demande (Loss of load évitée)"*
- *"Horizon de temps de la flexibilité, activer un prix"*
- *"Est-ce qu'il y a de la loss of load même avec de la flexibilité ?"*

If flexibility was activated, we examine the `operation_flexibility_power`
variable to see how much load was shifted and whether it reduced LOLE.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Check if flexibility was activated ────────────────────────────
flex_active = False
if inp_ds is not None:
    fd = inp_ds.get("flexibility_demand")
    if fd is not None and float(fd.values) > 0:
        flex_active = True
        print(f"✓ Flexibility is ACTIVE (demand fraction = {float(fd.values):.2%})")
    else:
        print("✗ Flexibility is INACTIVE (flexibility_demand = 0)")
        print("  → Re-run the model with flexibility parameters set (see Section 7.0.2)")

# Reload solution.nc to pick up new results
sol_path_check = DIAG_DIR / f"solution_{MODEL_YEAR}.nc"
if sol_path_check.exists():
    sol_ds_fresh = xr.open_dataset(sol_path_check)
else:
    sol_ds_fresh = sol_ds

# ── Look for flexibility-related variables ────────────────────────
FLEX_CANDIDATES = ["operation_flexibility_power", "flexibility_power",
                   "operation_flex_up", "operation_flex_down"]

flex_var = None
if sol_ds_fresh is not None:
    for v in sol_ds_fresh.data_vars:
        if "flex" in v.lower():
            print(f"  Found flexibility variable: {v} — dims={sol_ds_fresh[v].dims}, shape={sol_ds_fresh[v].shape}")
            if flex_var is None:
                flex_var = v

if flex_var and flex_active:
    flex_da = sol_ds_fresh[flex_var]
    print(f"\n── Flexibility dispatch: {flex_var} ──")

    fig, axes = plt.subplots(2, 1, figsize=(16, 10))

    flex_by_country = {}
    for area in COUNTRIES_MODELLED:
        try:
            f_area = flex_da.sel(area=area)
            if "resource" in f_area.dims:
                for r in f_area.coords["resource"].values:
                    if "electr" in str(r).lower():
                        f_area = f_area.sel(resource=r); break
            if "year_op" in f_area.dims:
                f_area = f_area.sel(year_op=MODEL_YEAR)
            vals = f_area.values.flatten()[:8760]
            shifted_gwh = abs(vals).sum() / 1000
            flex_by_country[area] = shifted_gwh
            if abs(vals).max() > 0:
                axes[0].plot(range(len(vals)), vals, label=f"{area} ({shifted_gwh:.1f} GWh)",
                           alpha=0.7, linewidth=0.5)
        except:
            pass

    axes[0].set_title(f"Hourly flexibility dispatch ({flex_var})")
    axes[0].set_ylabel("Power (MW)")
    axes[0].set_xlabel("Hour")
    axes[0].legend(ncol=3, fontsize=8)
    axes[0].axhline(y=0, color="gray", linewidth=0.5)

    # Bar chart
    countries = list(flex_by_country.keys())
    values = [flex_by_country[c] for c in countries]
    axes[1].bar(countries, values, color="steelblue")
    axes[1].set_ylabel("Total shifted energy (GWh)")
    axes[1].set_title("Annual flexibility activation by country")

    plt.tight_layout()
    plt.savefig(DIAG_DIR / "ev_flexibility.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Compare LOLE with/without flexibility ──
    print("\n── LOLE comparison (requires baseline without flex) ──")
    print("  Current LOLE with flexibility:")
    for area in COUNTRIES_MODELLED:
        r = adequacy_results.get(area, {})
        lole = r.get("lole_hours", "N/A")
        ens = r.get("ens_gwh", "N/A")
        print(f"    {area}: LOLE = {lole}h, ENS = {ens} GWh")
    print("\n  To measure 'loss of load évitée', compare with the baseline run")
    print("  (run with FLEX_DEMAND_FRACTION = 0 and save results separately)")

elif not flex_active:
    print("\n  ⚠ Flexibility parameters are all zero. The model has no demand flexibility.")
    print("  Set FLEX_DEMAND_FRACTION > 0 in Section 7.0.2 and re-solve to see flexibility results.")
else:
    print(f"\n  No flexibility variables found in solution.nc.")
    if sol_ds_fresh is not None:
        print(f"  Available vars: {sorted([v for v in sol_ds_fresh.data_vars if 'flex' in v.lower() or 'demand' in v.lower()])}")


## 16. Peak Power Management & Storage Adequacy

**Supervisor questions**:
- *"Gestion sur la puissance max"*
- *"Le stockage"*
- *"Capacité maximale"*
- *"Besoin de flexibilité plus long terme"*

Peak demand management: what happens during the top-50 peak hours?
Storage adequacy: are batteries / hydro / H2 storage correctly sized?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

LS_VAR = "operation_load_shedding_power"
CONV_VAR = "operation_conversion_power"

if sol_ds is not None and LS_VAR in sol_ds:
    ls_da = sol_ds[LS_VAR]

    # ── Top-50 stress hours analysis ─────────────────────────────
    print("── Top-50 system stress hours (highest load shedding) ──")

    # Aggregate load shedding across countries
    sys_ls = np.zeros(8760)
    country_ls = {}
    for area in COUNTRIES_MODELLED:
        try:
            ls_area = ls_da.sel(area=area)
            if "resource" in ls_area.dims:
                for r in ls_area.coords["resource"].values:
                    if "electr" in str(r).lower():
                        ls_area = ls_area.sel(resource=r); break
            if "year_op" in ls_area.dims:
                ls_area = ls_area.sel(year_op=MODEL_YEAR)
            vals = ls_area.values.flatten()[:8760]
            country_ls[area] = vals
            sys_ls[:len(vals)] += vals
        except:
            pass

    top50_idx = np.argsort(sys_ls)[-50:][::-1]
    print(f"  Peak system load shedding: {sys_ls.max():,.0f} MW at hour {sys_ls.argmax()}")
    print(f"  Top-50 stress hours span: hours {top50_idx.min()}-{top50_idx.max()}")
    print(f"  Top-50 by month:")

    # Month distribution of top-50 hours
    months = (top50_idx // 730).clip(0, 11) + 1  # approximate
    month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    for m in range(1, 13):
        count = (months == m).sum()
        if count > 0:
            print(f"    {month_names[m-1]}: {count} hours")

    # ── Dispatch during top-50 hours ──
    if CONV_VAR in sol_ds:
        conv_da = sol_ds[CONV_VAR]
        tech_dim = None
        for dim in conv_da.dims:
            if "tech" in dim.lower() or "conversion" in dim.lower():
                tech_dim = dim; break

        if tech_dim:
            all_techs = [str(c) for c in conv_da.coords[tech_dim].values]
            print(f"\n  Dispatch during top-50 stress hours (system-wide):")
            tech_during_stress = {}
            for tech in all_techs:
                total = 0
                for area in COUNTRIES_MODELLED:
                    try:
                        p = conv_da.sel({tech_dim: tech, "area": area})
                        if "resource" in p.dims:
                            for r in p.coords["resource"].values:
                                if "electr" in str(r).lower():
                                    p = p.sel(resource=r); break
                        if "year_op" in p.dims:
                            p = p.sel(year_op=MODEL_YEAR)
                        vals = p.values.flatten()[:8760]
                        total += vals[top50_idx].mean()
                    except:
                        pass
                if abs(total) > 1:
                    tech_during_stress[tech] = total
                    print(f"    {tech:30s}: avg {total:10,.0f} MW")

    # ── Storage state during stress ──
    STOR_VAR = "operation_storage_level"
    if STOR_VAR in sol_ds:
        stor_da = sol_ds[STOR_VAR]
        print(f"\n── Storage state during top-50 stress hours ──")
        for dim in stor_da.dims:
            if "tech" in dim.lower() or "storage" in dim.lower():
                for tech in [str(c) for c in stor_da.coords[dim].values]:
                    for area in COUNTRIES_MODELLED:
                        try:
                            soc = stor_da.sel({dim: tech, "area": area})
                            if "year_op" in soc.dims:
                                soc = soc.sel(year_op=MODEL_YEAR)
                            vals = soc.values.flatten()[:8760]
                            if vals.max() > 0:
                                stress_soc = vals[top50_idx]
                                capacity = vals.max()
                                fill_pct = stress_soc.mean() / capacity * 100 if capacity > 0 else 0
                                print(f"    {area}/{tech}: avg fill during stress = {fill_pct:.1f}% "
                                      f"(mean SoC = {stress_soc.mean():,.0f} / {capacity:,.0f} MWh)")
                        except:
                            pass
                break

    # ── Figure: Load shedding duration curve ──
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # System LS duration curve
    sorted_ls = np.sort(sys_ls)[::-1]
    axes[0].plot(range(len(sorted_ls)), sorted_ls / 1000, color="red", linewidth=1.5)
    axes[0].fill_between(range(len(sorted_ls)), sorted_ls / 1000, alpha=0.2, color="red")
    axes[0].set_xlabel("Hour (sorted by severity)")
    axes[0].set_ylabel("System load shedding (GW)")
    axes[0].set_title("Load Shedding Duration Curve (system)")
    axes[0].set_xlim(0, max(1, (sorted_ls > 0).sum() * 1.1))

    # Per-country LS sorted
    for area in COUNTRIES_MODELLED:
        if area in country_ls:
            vals = np.sort(country_ls[area])[::-1]
            if vals.max() > 0:
                n_pos = (vals > 0).sum()
                axes[1].plot(range(n_pos), vals[:n_pos] / 1000, label=area, linewidth=1)

    axes[1].set_xlabel("Hour (sorted)")
    axes[1].set_ylabel("Load shedding (GW)")
    axes[1].set_title("Load Shedding Duration Curve (per country)")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(DIAG_DIR / "peak_management.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Long-term flexibility needs ──
    print("\n── Long-term flexibility needs ──")
    print("  Consecutive stress hour streaks (system LS > 0):")
    is_stress = sys_ls > 0
    streaks = []
    current = 0
    for h in range(8760):
        if is_stress[h]:
            current += 1
        else:
            if current > 0:
                streaks.append((h - current, current))
            current = 0
    if current > 0:
        streaks.append((8760 - current, current))
    streaks.sort(key=lambda x: -x[1])
    for start, length in streaks[:10]:
        end = start + length
        day_start = start // 24
        day_end = end // 24
        print(f"    Hours {start}-{end} (day {day_start}-{day_end}): {length} consecutive hours")
    if streaks:
        print(f"  Longest stress event: {streaks[0][1]} hours ({streaks[0][1]/24:.1f} days)")
        print(f"  This defines the 'Dunkelflaute' duration the system must withstand.")


## 17. Final Adequacy Dashboard

Summary table and key conclusions for the CLEVER Sufficiency Scenario 2050.

In [ ]:
import pandas as pd

# ── Build summary DataFrame ──────────────────────────────────────
rows = []
for area in COUNTRIES_MODELLED:
    r = adequacy_results.get(area, {})
    row = {
        "Country": area,
        "ENS (GWh)": r.get("ens_gwh", 0),
        "LOLE (h)": r.get("lole_hours", 0),
        "Peak LS (MW)": r.get("peak_shedding_mw", 0),
        "Max dual (EUR/MWh)": r.get("dual_max", 0),
        "Hours at VoLL": r.get("dual_at_voll", 0),
        "Mean price (EUR/MWh)": r.get("mean_price", 0),
    }
    rows.append(row)

summary_df = pd.DataFrame(rows)

# Add totals row
totals = summary_df.select_dtypes(include="number").sum()
totals["Country"] = "TOTAL"
totals["Mean price (EUR/MWh)"] = summary_df["Mean price (EUR/MWh)"].mean()
summary_df = pd.concat([summary_df, pd.DataFrame([totals])], ignore_index=True)

print("=" * 100)
print("FINAL ADEQUACY DASHBOARD  —  CLEVER Sufficiency Scenario 2050")
print("=" * 100)
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:,.1f}"))

# ── Flexibility status ──
print("\n── Flexibility Status ──")
if inp_ds is not None:
    fd = inp_ds.get("flexibility_demand")
    if fd is not None:
        fd_val = float(fd.values)
        if fd_val > 0:
            print(f"  ✓ ACTIVE: {fd_val:.0%} of demand flexible, "
                  f"conservation window = {int(inp_ds['flexibility_conservation_hrs'].values)}h, "
                  f"cost = {float(inp_ds['flexibility_variable_cost'].values)} EUR/MWh")
        else:
            print("  ✗ INACTIVE: Set FLEX_DEMAND_FRACTION > 0 in Section 7.0.2")

# ── Key conclusions ──
total_ens = sum(r.get("ens_gwh", 0) or 0 for r in adequacy_results.values())
total_lole = sum(r.get("lole_hours", 0) or 0 for r in adequacy_results.values())
worst_country = max(adequacy_results.items(), key=lambda x: x[1].get("ens_gwh", 0) or 0)

print(f"\n── Key Findings ──")
print(f"  Total system ENS: {total_ens:,.1f} GWh ({total_ens/1000:.2f} TWh)")
print(f"  Total LOLE hours: {total_lole:,d}")
print(f"  Worst country: {worst_country[0]} ({worst_country[1].get('ens_gwh', 0):,.1f} GWh ENS, "
      f"{worst_country[1].get('lole_hours', 0)} LOLE hours)")

if total_ens > 0:
    print("\n  ⚠ ADEQUACY CONCERN: Significant load shedding persists.")
    print("    Possible mitigations:")
    print("    1. Activate/increase demand flexibility (EV smart charging)")
    print("    2. Raise CCGT H2 investment cap or add new dispatchable capacity")
    print("    3. Increase interconnection capacity (import from surplus countries)")
    print("    4. Add more battery/H2 storage for Dunkelflaute periods")
else:
    print("\n  ✓ System is adequate: no load shedding detected.")

# ── Save summary to CSV ──
summary_df.to_csv(DIAG_DIR / "adequacy_summary.csv", index=False)
print(f"\nSummary saved to {DIAG_DIR / 'adequacy_summary.csv'}")


---

## 18. Post-optimisation analysis — Dispatch & Flow Detail — Solution & Dual inspection

The following cells read the `solution_{year}.nc` and `dual_{year}.nc` files
saved by the solver, and produce publication-quality visualisations that
illustrate **how a 100 % renewable + sufficiency system maintains adequacy**
across the year.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# ── Reuse solution & dual datasets from Section 9 ────────────────
# sol_ds, dual_ds, inp_ds were loaded in the adequacy analysis cell (Section 9)
# Create aliases for backward compatibility
if 'sol_ds' in dir() and sol_ds is not None:
    sol = sol_ds
    print(f"Using sol_ds from Section 9: {len(sol_ds.data_vars)} variables")
else:
    sol_path = DIAG_DIR / f"solution_{MODEL_YEAR}.nc"
    sol = xr.open_dataset(sol_path) if sol_path.exists() else None
    sol_ds = sol
    print(f"Loaded solution.nc: {sol_path}")

if 'dual_ds' in dir() and dual_ds is not None:
    dual = dual_ds
else:
    dual_path = DIAG_DIR / f"dual_{MODEL_YEAR}.nc"
    dual = xr.open_dataset(dual_path) if dual_path.exists() else None
    dual_ds = dual

print(f"Solution vars: {len(sol.data_vars) if sol else 0}")
print(f"Dual vars: {len(dual.data_vars) if dual else 0}")


### 18.1 Extract hourly dispatch from POMMES results

Build a tidy DataFrame of hourly generation by technology and area,
which forms the basis for all supply-demand visualisations.

In [ ]:
def _guess_dims(da, kind):
    """Heuristic to identify dimension names in an xarray DataArray."""
    mapping = {}
    for d in da.dims:
        dl = d.lower()
        if any(k in dl for k in ["area", "zone", "country", "node"]):
            mapping["area"] = d
        elif any(k in dl for k in ["hour", "time", "step"]):
            mapping["hour"] = d
        elif any(k in dl for k in ["resource", "commodity", "carrier"]):
            mapping["resource"] = d
        elif any(k in dl for k in ["year_op", "year"]):
            mapping["year"] = d
        elif kind == "conversion" and any(k in dl for k in ["conversion_tech", "tech"]):
            mapping["tech"] = d
        elif kind == "storage" and any(k in dl for k in ["storage_tech", "tech"]):
            mapping["tech"] = d
        elif kind == "transport" and any(k in dl for k in ["transport", "link", "line"]):
            mapping["link"] = d
    return mapping


def extract_hourly_dispatch(model, sol_ds, year_op: int) -> pd.DataFrame:
    """Extract hourly dispatch (MW) from POMMES results, with sol.nc fallback."""
    # Attempt 1: model API
    df = pd.DataFrame()
    try:
        gen = model.get_results("operation", "conversion_power")
        df = gen.to_pandas() if hasattr(gen, "to_pandas") else pd.DataFrame(gen)
    except Exception:
        try:
            gen = model.get_results("operation", "production")
            df = gen.to_pandas() if hasattr(gen, "to_pandas") else pd.DataFrame(gen)
        except Exception:
            pass

    # Attempt 2: solution.nc fallback
    if df.empty and sol_ds is not None:
        CONV_VAR = "operation_conversion_power"
        if CONV_VAR in sol_ds:
            conv_da = sol_ds[CONV_VAR]
            dims = _guess_dims(conv_da, "conversion")
            rows = []
            tech_dim = dims.get("tech")
            if tech_dim:
                for tech in conv_da.coords[tech_dim].values:
                    for area in COUNTRIES_MODELLED:
                        try:
                            sel = {tech_dim: tech}
                            if "area" in dims:
                                sel[dims["area"]] = area
                            p = conv_da.sel(sel)
                            if "resource" in dims:
                                for r in p.coords[dims["resource"]].values:
                                    if "electr" in str(r).lower():
                                        p = p.sel({dims["resource"]: r}); break
                            if "year" in dims:
                                p = p.sel({dims["year"]: year_op})
                            vals = p.values.flatten()[:8760]
                            for h, v in enumerate(vals):
                                if abs(v) > 0.01:
                                    rows.append({"hour": h, "area": area, "tech": str(tech),
                                                "resource": "electricity", "value": float(v)})
                        except:
                            pass
            if rows:
                df = pd.DataFrame(rows)
                print(f"  Dispatch extracted from solution.nc ({len(df)} rows)")

    if not df.empty:
        # Normalize columns
        df.columns = [str(c).strip() for c in df.columns]
        rename = {}
        for cand in ["t", "time", "step", "hour_op"]:
            if "hour" not in df.columns and cand in df.columns:
                rename[cand] = "hour"; break
        for cand in ["conversion_power", "power", "production"]:
            if "value" not in df.columns and cand in df.columns:
                rename[cand] = "value"; break
        for cand in ["conversion_tech", "technology", "name"]:
            if "tech" not in df.columns and cand in df.columns:
                rename[cand] = "tech"; break
        for cand in ["zone", "country", "node"]:
            if "area" not in df.columns and cand in df.columns:
                rename[cand] = "area"; break
        for cand in ["commodity", "carrier"]:
            if "resource" not in df.columns and cand in df.columns:
                rename[cand] = "resource"; break
        if rename:
            df = df.rename(columns=rename)
        if "area" not in df.columns:
            df["area"] = "UNKNOWN"
        if "resource" not in df.columns:
            df["resource"] = "electricity"
        df["area"] = df["area"].astype(str).str.upper().str.strip()
        df["resource"] = df["resource"].astype(str).str.lower().str.strip()
        df = df[df["resource"] == "electricity"].copy()
        if "value" in df.columns:
            df["value"] = pd.to_numeric(df["value"], errors="coerce").fillna(0.0)
        if "hour" in df.columns:
            df["hour"] = pd.to_numeric(df["hour"], errors="coerce").astype(int)

    return df

dispatch_df = extract_hourly_dispatch(model, sol_ds, MODEL_YEAR)
print(f"Dispatch DataFrame: {len(dispatch_df)} rows")
if not dispatch_df.empty and "tech" in dispatch_df.columns:
    print(f"Technologies: {sorted(dispatch_df['tech'].unique())}")
    print(f"Areas: {sorted(dispatch_df['area'].unique())}")


### 18.2 Stacked dispatch (supply vs. demand) — system aggregate

The core adequacy visualisation: a stacked area chart showing how each
technology contributes to meeting demand hour by hour across the full year.

In [ ]:
# ── Technology colour palette (consistent across all plots) ─────────
TECH_COLORS = {
    "Solar": "#FFD700",
    "Wind_Onshore": "#4CAF50",
    "Wind_Offshore": "#1565C0",
    "RoR_Hydro": "#00BCD4",
    "Reservoir_Hydro_Plant": "#0097A7",
    "Gas": "#FF5722",
    "Hydrogen_power_plant": "#9C27B0",
    "Biomass": "#795548",
    "Nuclear": "#E91E63",
    "Coal": "#424242",
    "Waste": "#607D8B",
    "Other": "#9E9E9E",
    "Load_Shedding": "#F44336",
}
TECH_ORDER = list(TECH_COLORS.keys())

def plot_system_dispatch(dispatch_df, demand_dict, countries, year, focus_week=None):
    """
    Plot stacked dispatch for the system (sum over all countries).
    If focus_week is set (1-52), zoom on that week; otherwise show full year.
    """
    if dispatch_df.empty or "tech" not in dispatch_df.columns:
        print("No dispatch data to plot.")
        return

    # Filter to modelled countries
    df = dispatch_df[dispatch_df["area"].isin(countries)].copy()

    # Pivot: rows=hour, cols=tech, values=sum over areas
    piv = df.pivot_table(index="hour", columns="tech", values="value", aggfunc="sum").fillna(0)
    piv = piv.reindex(columns=[t for t in TECH_ORDER if t in piv.columns], fill_value=0)

    # System demand
    sys_demand = np.zeros(8760)
    for c in countries:
        d = demand_dict.get(c)
        if d is not None:
            vals = d["demand"].to_numpy() if hasattr(d["demand"], "to_numpy") else np.array(d["demand"].to_list())
            sys_demand[:len(vals)] += vals[:8760]

    hours = np.arange(8760)
    dt_index = pd.date_range(f"{year}-01-01", periods=8760, freq="h")

    if focus_week is not None:
        h0 = (focus_week - 1) * 168
        h1 = min(h0 + 168, 8760)
        mask = (hours >= h0) & (hours < h1)
        title_suffix = f" — Week {focus_week}"
    else:
        mask = np.ones(8760, dtype=bool)
        title_suffix = " — Full year"

    fig, ax = plt.subplots(figsize=(16, 6))

    # Stacked area
    bottom = np.zeros(mask.sum())
    x = dt_index[mask]
    for tech in piv.columns:
        vals = piv[tech].reindex(hours, fill_value=0).values[mask]
        vals = np.clip(vals, 0, None)  # only positive generation
        color = TECH_COLORS.get(tech, "#BDBDBD")
        ax.fill_between(x, bottom, bottom + vals, label=tech, color=color, alpha=0.85, linewidth=0)
        bottom += vals

    # Demand line
    ax.plot(x, sys_demand[mask], color="black", linewidth=1.2, label="Demand", zorder=10)

    ax.set_ylabel("Power (MW)")
    ax.set_title(f"System dispatch — CLEVER {year} ({len(countries)} countries){title_suffix}")
    ax.legend(loc="upper right", fontsize=8, ncol=2, framealpha=0.9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.set_xlim(x[0], x[-1])
    ax.set_ylim(0, None)
    plt.tight_layout()
    plt.show()

# Full-year view
plot_system_dispatch(dispatch_df, demand_dict, COUNTRIES_MODELLED, MODEL_YEAR)

# Winter peak week (typically week 3-5) and summer trough (week 30-32)
plot_system_dispatch(dispatch_df, demand_dict, COUNTRIES_MODELLED, MODEL_YEAR, focus_week=4)
plot_system_dispatch(dispatch_df, demand_dict, COUNTRIES_MODELLED, MODEL_YEAR, focus_week=30)

### 18.3 Price duration curves

Marginal electricity prices sorted from highest to lowest reveal the scarcity
structure. In an adequate renewable system, prices should be near zero most of
the time (VRE marginal cost) with occasional peaks during low-wind/low-sun periods.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Left panel: full price duration curve ──────────────────────
ax = axes[0]
for area in sorted(COUNTRIES_MODELLED):
    ap = prices_df[prices_df["area"] == area]["value"].values
    ap_sorted = np.sort(ap)[::-1]
    x_pct = np.linspace(0, 100, len(ap_sorted))
    ax.plot(x_pct, ap_sorted, label=area, linewidth=1.0)

ax.set_xlabel("% of hours")
ax.set_ylabel("Marginal price (EUR/MWh)")
ax.set_title("Price duration curves — full range")
ax.legend(fontsize=8, ncol=2)
ax.set_xlim(0, 100)
ax.grid(True, alpha=0.3)

# ── Right panel: zoom on 0–200 EUR/MWh (normal operating range) ──
ax = axes[1]
for area in sorted(COUNTRIES_MODELLED):
    ap = prices_df[prices_df["area"] == area]["value"].values
    ap_sorted = np.sort(ap)[::-1]
    x_pct = np.linspace(0, 100, len(ap_sorted))
    ax.plot(x_pct, ap_sorted, label=area, linewidth=1.0)

ax.set_xlabel("% of hours")
ax.set_ylabel("Marginal price (EUR/MWh)")
ax.set_title("Price duration curves — zoom on normal range")
ax.set_ylim(-10, 200)
ax.set_xlim(0, 100)
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Summary statistics ─────────────────────────────────────────
print("Price statistics (EUR/MWh):")
for area in sorted(COUNTRIES_MODELLED):
    ap = prices_df[prices_df["area"] == area]["value"].values
    zero_hours = (ap <= 0.01).sum()
    print(f"  {area}: mean={np.mean(ap):.1f} | median={np.median(ap):.1f} | "
          f"max={np.max(ap):.0f} | zero-price hours={zero_hours} ({100*zero_hours/len(ap):.0f}%)")

### 18.4 Storage dynamics

State of charge and charge/discharge patterns for battery and hydro storage.
These reveal how flexibility assets bridge VRE intermittency.

In [ ]:
def extract_storage_soc(model, sol_ds, year_op: int) -> pd.DataFrame:
    """Extract storage state-of-charge (MWh), with sol.nc fallback."""
    df = pd.DataFrame()
    # Attempt 1: model API
    candidates = [
        ("operation", "storage_level"),
        ("operation", "energy_level"),
        ("operation", "state_of_charge"),
    ]
    for rt, rn in candidates:
        try:
            soc = model.get_results(rt, rn)
            df = soc.to_pandas() if hasattr(soc, "to_pandas") else pd.DataFrame(soc)
            if not df.empty:
                break
        except Exception:
            continue

    # Attempt 2: sol.nc fallback
    if df.empty and sol_ds is not None:
        STOR_VAR = "operation_storage_level"
        if STOR_VAR in sol_ds:
            stor_da = sol_ds[STOR_VAR]
            dims = _guess_dims(stor_da, "storage")
            tech_dim = dims.get("tech")
            rows = []
            if tech_dim:
                for tech in stor_da.coords[tech_dim].values:
                    for area in COUNTRIES_MODELLED:
                        try:
                            sel = {tech_dim: tech}
                            if "area" in dims:
                                sel[dims["area"]] = area
                            s = stor_da.sel(sel)
                            if "year" in dims:
                                s = s.sel({dims["year"]: year_op})
                            vals = s.values.flatten()[:8760]
                            for h, v in enumerate(vals):
                                rows.append({"hour": h, "area": area, "tech": str(tech),
                                           "value": float(v)})
                        except:
                            pass
            if rows:
                df = pd.DataFrame(rows)
                print(f"  Storage SoC extracted from solution.nc ({len(df)} rows)")

    if not df.empty:
        df.columns = [str(c).strip() for c in df.columns]
        rename = {}
        for cand in ["storage_tech", "technology", "name"]:
            if "tech" not in df.columns and cand in df.columns:
                rename[cand] = "tech"; break
        for cand in ["zone", "country", "node"]:
            if "area" not in df.columns and cand in df.columns:
                rename[cand] = "area"; break
        for cand in ["storage_level", "energy_level", "soc"]:
            if "value" not in df.columns and cand in df.columns:
                rename[cand] = "value"; break
        if rename:
            df = df.rename(columns=rename)
    return df

soc_df = extract_storage_soc(model, sol_ds, MODEL_YEAR)
print(f"Storage SoC DataFrame: {len(soc_df)} rows")
if not soc_df.empty and "tech" in soc_df.columns:
    for tech in sorted(soc_df["tech"].unique()):
        sub = soc_df[soc_df["tech"] == tech]
        print(f"  {tech}: max SoC = {sub['value'].max():,.0f} MWh")


### 18.5 Cross-border electricity flows

Net annual flows between interconnected countries. Positive values indicate
net export from the first country to the second.

In [ ]:
def extract_flows(model, sol_ds, year_op: int) -> pd.DataFrame:
    """Extract hourly cross-border electricity flows, with sol.nc fallback."""
    df = pd.DataFrame()
    candidates = [
        ("operation", "transport_power"),
        ("operation", "flow"),
        ("operation", "link_flow"),
    ]
    for rt, rn in candidates:
        try:
            fl = model.get_results(rt, rn)
            df = fl.to_pandas() if hasattr(fl, "to_pandas") else pd.DataFrame(fl)
            if not df.empty:
                break
        except Exception:
            continue

    # Attempt 2: sol.nc fallback
    if df.empty and sol_ds is not None:
        FLOW_VAR = "operation_transport_power"
        if FLOW_VAR in sol_ds:
            flow_da = sol_ds[FLOW_VAR]
            dims = _guess_dims(flow_da, "transport")
            link_dim = dims.get("link")
            rows = []
            if link_dim:
                for link in flow_da.coords[link_dim].values:
                    try:
                        f = flow_da.sel({link_dim: link})
                        if "resource" in dims:
                            for r in f.coords[dims["resource"]].values:
                                if "electr" in str(r).lower():
                                    f = f.sel({dims["resource"]: r}); break
                        if "year" in dims:
                            f = f.sel({dims["year"]: year_op})
                        vals = f.values.flatten()[:8760]
                        for h, v in enumerate(vals):
                            if abs(v) > 0.01:
                                rows.append({"hour": h, "link": str(link),
                                           "resource": "electricity", "value": float(v)})
                    except:
                        pass
            if rows:
                df = pd.DataFrame(rows)
                print(f"  Flows extracted from solution.nc ({len(df)} rows)")

    if not df.empty:
        df.columns = [str(c).strip() for c in df.columns]
    return df

flow_df = extract_flows(model, sol_ds, MODEL_YEAR)
print(f"Flow DataFrame: {len(flow_df)} rows")
if not flow_df.empty and "link" in flow_df.columns:
    print(f"Links: {sorted(flow_df['link'].unique())}")


### 18.6 Per-country annual energy balance

How each country meets its electricity demand: a stacked bar chart of
annual generation by technology (TWh) versus demand.

In [ ]:
if not dispatch_df.empty and "tech" in dispatch_df.columns:
    # Annual generation by area and tech (TWh)
    df = dispatch_df[dispatch_df["area"].isin(COUNTRIES_MODELLED)].copy()
    annual = df.groupby(["area", "tech"])["value"].sum().reset_index()
    annual["value_twh"] = annual["value"] / 1e6  # MW·h → TWh

    piv = annual.pivot_table(index="area", columns="tech", values="value_twh", aggfunc="sum").fillna(0)
    piv = piv.reindex(columns=[t for t in TECH_ORDER if t in piv.columns], fill_value=0)

    # Demand for comparison
    demand_twh = {}
    for c in COUNTRIES_MODELLED:
        d = demand_dict.get(c)
        if d is not None:
            vals = d["demand"].to_numpy() if hasattr(d["demand"], "to_numpy") else np.array(d["demand"].to_list())
            demand_twh[c] = float(np.sum(vals)) / 1e6

    fig, ax = plt.subplots(figsize=(14, 6))

    bottom = np.zeros(len(piv))
    for tech in piv.columns:
        vals = piv[tech].values
        color = TECH_COLORS.get(tech, "#BDBDBD")
        ax.bar(piv.index, vals, bottom=bottom, label=tech, color=color, alpha=0.85)
        bottom += vals

    # Overlay demand markers
    for i, area in enumerate(piv.index):
        if area in demand_twh:
            ax.plot(i, demand_twh[area], "kD", markersize=8, zorder=10)

    ax.set_ylabel("Annual energy (TWh)")
    ax.set_title(f"Per-country energy balance — CLEVER {MODEL_YEAR} (diamonds = demand)")
    ax.legend(loc="upper right", fontsize=8, ncol=2, framealpha=0.9)
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()
else:
    print("No dispatch data available for per-country energy balance.")

### 18.7 Residual load duration curve

The residual load (demand minus VRE generation) shows the burden placed on
dispatchable technologies and flexibility assets. In a well-designed renewable
system, the residual load has many negative hours (VRE surplus) and a steep
but short positive tail (scarcity moments requiring storage or dispatchable backup).

In [ ]:
VRE_TECHS = {"Solar", "Wind_Onshore", "Wind_Offshore"}

if not dispatch_df.empty and "tech" in dispatch_df.columns:
    fig, ax = plt.subplots(figsize=(14, 5))

    for area in sorted(COUNTRIES_MODELLED):
        # VRE generation
        vre = dispatch_df[(dispatch_df["area"] == area) & (dispatch_df["tech"].isin(VRE_TECHS))]
        vre_hourly = vre.groupby("hour")["value"].sum().reindex(range(8760), fill_value=0).values

        # Demand
        d = demand_dict.get(area)
        if d is None:
            continue
        demand_vals = d["demand"].to_numpy() if hasattr(d["demand"], "to_numpy") else np.array(d["demand"].to_list())
        demand_vals = demand_vals[:8760]

        residual = demand_vals - vre_hourly[:len(demand_vals)]
        residual_sorted = np.sort(residual)[::-1]
        x_pct = np.linspace(0, 100, len(residual_sorted))
        ax.plot(x_pct, residual_sorted / 1e3, label=area, linewidth=1.0)

    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
    ax.set_xlabel("% of hours (sorted)")
    ax.set_ylabel("Residual load (GW)")
    ax.set_title(f"Residual load duration curves (demand - VRE) — CLEVER {MODEL_YEAR}")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Print key statistics
    print("\nResidual load statistics (GW):")
    for area in sorted(COUNTRIES_MODELLED):
        vre = dispatch_df[(dispatch_df["area"] == area) & (dispatch_df["tech"].isin(VRE_TECHS))]
        vre_hourly = vre.groupby("hour")["value"].sum().reindex(range(8760), fill_value=0).values
        d = demand_dict.get(area)
        if d is None:
            continue
        demand_vals = d["demand"].to_numpy() if hasattr(d["demand"], "to_numpy") else np.array(d["demand"].to_list())
        residual = demand_vals[:8760] - vre_hourly[:len(demand_vals)]
        neg_hours = (residual < 0).sum()
        print(f"  {area}: peak={np.max(residual)/1e3:.1f} GW | min={np.min(residual)/1e3:.1f} GW | "
              f"surplus hours={neg_hours} ({100*neg_hours/len(residual):.0f}%)")
else:
    print("No dispatch data available.")

### 18.8 Hourly price heatmaps (hour-of-day vs day-of-year)

These heatmaps reveal the temporal structure of scarcity in a renewable
system: winter evenings (no solar, high heating demand) vs summer middays
(solar surplus, near-zero prices).

In [ ]:
# Select 3 representative countries for the heatmap
HEATMAP_COUNTRIES = [c for c in ["FR", "DE", "ES"] if c in COUNTRIES_MODELLED]

if HEATMAP_COUNTRIES:
    fig, axes = plt.subplots(1, len(HEATMAP_COUNTRIES), figsize=(6 * len(HEATMAP_COUNTRIES), 5))
    if len(HEATMAP_COUNTRIES) == 1:
        axes = [axes]

    for ax, area in zip(axes, HEATMAP_COUNTRIES):
        ap = prices_df[prices_df["area"] == area].sort_values("hour")
        if ap.empty:
            continue

        # Build 365 x 24 matrix
        vals = ap["value"].values[:8760]
        n = min(len(vals), 8760)
        n_days = n // 24
        mat = vals[:n_days * 24].reshape(n_days, 24).T

        # Cap for visualisation (avoid load shedding dominating colourbar)
        vmax = min(np.percentile(mat, 99.5), 300)
        im = ax.imshow(mat, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=vmax,
                       interpolation="nearest", origin="lower")
        ax.set_xlabel("Day of year")
        ax.set_ylabel("Hour of day")
        ax.set_title(f"{area} — Hourly prices (EUR/MWh)")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.suptitle(f"Price heatmaps — CLEVER {MODEL_YEAR}", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No countries available for heatmap.")

### 18.9 Capacity utilisation and adequacy summary

Final overview: installed capacities, average utilisation rates, and
key adequacy metrics for the CLEVER sufficiency scenario.

In [ ]:
# ── Capacity utilisation (capacity factor) by technology ──────────
if not dispatch_df.empty and not conv_cap_df.empty:
    # Annual generation by tech (MWh)
    gen = dispatch_df[dispatch_df["area"].isin(COUNTRIES_MODELLED)].groupby("tech")["value"].sum()

    # Installed capacity by tech (MW)
    cap = conv_cap_df[conv_cap_df["area"].isin(COUNTRIES_MODELLED)].groupby("name")["value"].sum()

    util_rows = []
    for tech in sorted(set(gen.index) | set(cap.index)):
        gen_mwh = gen.get(tech, 0)
        cap_mw = cap.get(tech, 0)
        cf = gen_mwh / (cap_mw * 8760) * 100 if cap_mw > 0 else 0
        util_rows.append({
            "Technology": tech,
            "Capacity (GW)": cap_mw / 1e3,
            "Generation (TWh)": gen_mwh / 1e6,
            "Capacity Factor (%)": cf,
        })

    util_df = pd.DataFrame(util_rows).set_index("Technology").round(1)
    util_df = util_df[util_df["Capacity (GW)"] > 0].sort_values("Generation (TWh)", ascending=False)
    display(util_df)
else:
    print("No dispatch/capacity data for utilisation analysis.")

# ── Final adequacy statement ──────────────────────────────────────
print("\n" + "=" * 70)
print(f"  CLEVER {MODEL_YEAR} — ADEQUACY SUMMARY")
print("=" * 70)
total_demand_twh = sum(
    float(np.sum(d["demand"].to_numpy() if hasattr(d["demand"], "to_numpy") else np.array(d["demand"].to_list()))) / 1e6
    for c, d in demand_dict.items() if c in COUNTRIES_MODELLED
)
print(f"  Countries modelled:   {len(COUNTRIES_MODELLED)}")
print(f"  Total demand:         {total_demand_twh:,.0f} TWh")
if 'adeq_df' in dir() and not adeq_df.empty:
    print(f"  System LOLE:          {adeq_df['LOLE_hours'].sum():.0f} hours")
    print(f"  System ENS:           {adeq_df['ENS_GWh'].sum():.1f} GWh")
    countries_with_shedding = (adeq_df["LOLE_hours"] > 0).sum()
    if countries_with_shedding == 0:
        print("  Verdict:              FULLY ADEQUATE")
    else:
        print(f"  Verdict:              {countries_with_shedding} countries with load shedding")
print("=" * 70)